# Vom GuppyLM zum Vision Transformer — Kleidung erkennen

Im GuppyLM wird aus einer Folge von Texttokens eine Vorhersage für das nächste
Token. Hier wird aus einer Folge von **Bildpatches** eine Vorhersage für das
gesamte Bild: Welche Art Kleidungsstück zeigt es?

Wir bauen einen kleinen, vollständigen Vision Transformer (ViT) für
**Fashion-MNIST**. Q/K/V-Projektionen, mehrere Attentionköpfe, GELU-FFN,
Pre-Norm und Residualverbindungen bleiben wie im GuppyLM. Neu sind die
Patchprojektion und der Klassifikationskopf. Eine kausale Maske brauchen wir
für diese Aufgabe nicht: Das ganze Bild liegt bereits vor.

**So wird dieses Notebook gelesen:** Lesen Sie jeden Erklärabschnitt **vor**
der direkt folgenden Codezelle. Der Text führt deren Namen, Formen und Schritte
ein. Führen Sie dann die Zelle aus und vergleichen Sie die Beobachtung mit Ihrer
Erwartung. Ein neuer Begriff wird nicht erst hinter dem Code erklärt.

| Weg durch das Notebook | Was Sie dabei verstehen |
|---|---|
| 1. Umgebung | Hardware, Architekturwerte und Wiederholbarkeit |
| 2. Bilder und Ziele | Klassen, Batches und getrennte Datensplits |
| 3. Bild zu Sequenz | Patchzerlegung, Projektion, `[CLS]` und Position |
| 4. Transformer | Attention, FFN, Residualpfade und vollständiges Modell |
| 5. Lernen | Kreuzentropie, Validierung und Gewichtsupdates |
| 6. Prüfen und Anwenden | Bestes Modell laden, Abschlusstest und Vorhersagen |

**Voraussetzungen:** Python, Tensoren und die Trainingsschleife aus dem
GuppyLM-Notebook. Arbeiten Sie von oben nach unten mit dem Notebookordner als
Arbeitsverzeichnis. Trainingszeit und erreichbare Genauigkeit müssen auf dem
jeweiligen Gerät gemessen werden. Die Startkonfiguration ist keine garantierte
optimale Einstellung. Dieses Notebook enthält keine erfundenen Modellausgaben.

## 1. Umgebung

Die folgende Zelle installiert die Pakete in den **aktiven Notebook-Kernel**.
`torch` liefert Tensoren und automatische Differenzierung, `torchvision` den
Fashion-MNIST-Datenlader und die Bildtransformationen, `matplotlib` die Diagramme.
Die Datei `requirements-vit.txt` hält die dazugehörigen Paketversionen fest.
Die Installation braucht Internet; bei bereits eingerichteter Umgebung können
Sie die Zelle überspringen. Nach Änderungen an einer bereits importierten
PyTorch-Installation starten Sie den Kernel neu.

In [ ]:
%pip install -q -r requirements-vit.txt

### Arbeitsordner, Hardware und Zufall

`WORK` bezeichnet den Notebookordner. Fashion-MNIST wird später unter
`Data/FashionMNIST` abgelegt; eigene Gewichte landen in `Checkpoints`.
`DEVICE` wählt zuerst CUDA, dann Apples MPS, sonst die CPU.

`SEED` und `torch.manual_seed` legen den Start des Zufallsgenerators fest.
Für Datensplit und Batchreihenfolge verwenden wir später zusätzlich eigene
Generatoren. Das erleichtert Vergleiche; bitgleiche Ergebnisse über verschiedene
Geräte und Paketversionen sind dadurch nicht garantiert.

`HERUNTERLADEN = True` erlaubt beim ersten Datenladen den Download. Liegen die
Daten bereits lokal vor, werden sie wiederverwendet. Für einen vorbereiteten
Unterricht ohne Internet stellen Sie den Wert auf `False`. Wir rechnen in
float32; besondere Beschleunigungstechniken verdecken hier keine Rechenschritte.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

WORK = Path.cwd()
(WORK / "Checkpoints").mkdir(exist_ok=True)
SEED = 42
torch.manual_seed(SEED)
HERUNTERLADEN = True

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"PyTorch {torch.__version__}; Gerät: {DEVICE}")
print(f"Arbeitsverzeichnis: {WORK}")

### Die Architektur einstellen

Wie `GuppyConfig` sammelt `ViTConfig` die Architekturwerte an einer Stelle.
`image_size = 28` und `in_channels = 1` beschreiben die unveränderten
Graustufenbilder. `patch_size = 4` teilt jede Bildkante in sieben Abschnitte.
`n_patches` berechnet deshalb 49 Bildpatches. Mit einem zusätzlichen
Classification Token ergibt `seq_len` die Sequenzlänge 50.

Die aus GuppyLM bekannten Namen behalten ihre Bedeutung: `d_model = 128` ist
die Vektorbreite, `n_layers = 4` die Zahl aufeinanderfolgender Blöcke und
`n_heads = 4` die Zahl paralleler Köpfe je Block. Jeder Kopf hat 32 Dimensionen.
`ffn_hidden = 512` ist die innere FFN-Breite. `n_classes = 10` ersetzt am
Ausgang die Vokabulargröße. `dropout` ist nur im Trainingsmodus aktiv.

`__post_init__` weist unpassende Einstellungen zurück, bevor ein Rechenlauf
beginnt. Die mit `@property` markierten Werte werden aus der Konfiguration
berechnet und nicht separat eingestellt. Ändern Sie Architekturwerte vor einem
neuen Lauf hier und führen Sie anschließend das Notebook von vorn aus.

In [ ]:
@dataclass
class ViTConfig:
    image_size: int = 28
    in_channels: int = 1
    patch_size: int = 4
    d_model: int = 128
    n_layers: int = 4
    n_heads: int = 4
    ffn_hidden: int = 512
    n_classes: int = 10
    dropout: float = 0.1

    def __post_init__(self):
        for name in ("image_size", "in_channels", "patch_size", "d_model",
                     "n_layers", "n_heads", "ffn_hidden", "n_classes"):
            wert = getattr(self, name)
            if type(wert) is not int or wert <= 0:
                raise ValueError(f"{name} muss eine positive ganze Zahl sein.")
        if self.image_size % self.patch_size:
            raise ValueError("Die Bildkante muss durch patch_size teilbar sein.")
        if self.d_model % self.n_heads:
            raise ValueError("d_model muss durch n_heads teilbar sein.")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout muss zwischen 0 (inklusive) und 1 liegen.")

    @property
    def n_patches(self):
        return (self.image_size // self.patch_size) ** 2

    @property
    def seq_len(self):
        return self.n_patches + 1


konfig = ViTConfig()
print(konfig)
print(f"{konfig.n_patches} Bildpatches + [CLS] = {konfig.seq_len} Positionen")
print(f"Dimension je Kopf: {konfig.d_model // konfig.n_heads}")

## 2. Bilder und Ziele

Fashion-MNIST enthält 60.000 Trainingsbilder und 10.000 offizielle Testbilder.
Jedes Bild ist 28 × 28 Pixel groß und gehört zu genau einer von zehn Klassen.
Die Liste `KLASSEN` verwendet deutsche Namen **in der vorgegebenen ID-Reihenfolge**.
Beispielsweise bedeutet die ID 1 immer „Hose“.

`FashionMNIST(..., train=True)` öffnet zunächst ausschließlich den Trainingspool.
Den offiziellen Testsatz öffnen wir erst nach der Modellwahl in Abschnitt 6.
`root` ist der übergeordnete Datenordner; der Lader ergänzt `FashionMNIST` selbst.

`vorverarbeitung` beschreibt zwei feste Rechenschritte:

1. `ToTensor()` macht aus einem Bild einen float32-Tensor der Form
   `(1, 28, 28)` und teilt die ursprünglichen Werte 0 bis 255 durch 255.
2. `Normalize((0.5,), (0.5,))` berechnet `(Wert − 0.5) / 0.5`.
   Die Werte liegen anschließend zwischen −1 und 1. Die beiden 0,5 sind fest
   gewählte Konstanten, keine aus dem Testsatz geschätzten Statistiken.

Es gibt kein Resize, keine Umwandlung zu RGB und im Grundversuch keine zufällige
Bildaugmentation. Training und spätere Inferenz verwenden dieselbe Umrechnung.
Für diesen Fashion-MNIST-Datenweg prüfen wir die feste Bildform und Klassenzahl.

In [ ]:
from torchvision.datasets import FashionMNIST
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

if (konfig.image_size, konfig.in_channels, konfig.n_classes) != (28, 1, 10):
    raise ValueError("Fashion-MNIST benötigt image_size=28, in_channels=1, n_classes=10.")

KLASSEN = ["T-Shirt/Top", "Hose", "Pullover", "Kleid", "Mantel",
           "Sandale", "Hemd", "Sneaker", "Tasche", "Stiefelette"]
VORVERARBEITUNG = {"pixelteiler": 255.0, "mittel": 0.5, "standard": 0.5}
vorverarbeitung = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((VORVERARBEITUNG["mittel"],),
                         (VORVERARBEITUNG["standard"],)),
])

datenpool = FashionMNIST(root=WORK / "Data", train=True,
                       download=HERUNTERLADEN, transform=vorverarbeitung)
print(f"Trainingspool: {len(datenpool):,} Bilder")
for nummer, name in enumerate(KLASSEN):
    print(f"{nummer}: {name}")

### Training, Validierung und Test unterscheiden

Von den 60.000 Bildern reservieren wir **500 je Klasse** für die Validierung.
So entstehen 55.000 Trainings- und 5.000 Validierungsbilder mit gleicher
Klassenverteilung. Nur die 55.000 Trainingsbilder erzeugen Gewichtsupdates.
Die Validierung bestimmt später den besten Checkpoint. Die 10.000 offiziellen
Testbilder bleiben von beiden Aufgaben getrennt.

`torch.where` findet für jede Klasse ihre Indizes. `randperm` mischt diese mit
einem eigenen, durch `SEED` gestarteten Generator. Die ersten 500 Indizes gehen
in `val_ids`, der Rest in `train_ids`. `Subset` verweist auf diese ausgewählten
Bilder, ohne einen zweiten Bildbestand anzulegen. Die Assertions prüfen, dass
die beiden Indexmengen getrennt sind und zusammen den Pool vollständig abdecken.

Ein **Batch** enthält mehrere Bilder samt Klassen-IDs. `DataLoader` stellt
diese Batches bereit: `shuffle=True` mischt beim Training jede Epoche neu;
die Validierung bleibt in fester Reihenfolge. Eine Epoche ist ein Durchlauf über
alle Trainingsbilder. `num_workers=0` vermeidet zusätzliche Prozesse und ist
für dieses lokale Notebook auch unter Windows unkompliziert.

In [ ]:
BATCH_SIZE = 128
split_generator = torch.Generator().manual_seed(SEED)
train_ids, val_ids = [], []
for klasse in range(len(KLASSEN)):
    ids = torch.where(datenpool.targets == klasse)[0]
    ids = ids[torch.randperm(len(ids), generator=split_generator)]
    val_ids.extend(ids[:500].tolist())
    train_ids.extend(ids[500:].tolist())

assert not (set(train_ids) & set(val_ids))
assert set(train_ids) | set(val_ids) == set(range(len(datenpool)))
train_daten = Subset(datenpool, train_ids)
val_daten = Subset(datenpool, val_ids)

def baue_loader(train_daten, val_daten, batch_size):
    generator = torch.Generator().manual_seed(SEED + 1)
    train_loader = DataLoader(train_daten, batch_size=batch_size, shuffle=True,
                              generator=generator, num_workers=0)
    val_loader = DataLoader(val_daten, batch_size=batch_size, shuffle=False,
                            num_workers=0)
    return train_loader, val_loader

train_loader, val_loader = baue_loader(train_daten, val_daten, BATCH_SIZE)
print(f"Training: {len(train_daten):,}; Validierung: {len(val_daten):,}")

### Beobachtung: Bilder und Zielwerte

`x_probe` sammelt ein Trainingsbild je Klasse, `y_probe` deren Klassen-IDs.
Die Splitzelle hat die Trainingsindizes klassenweise gesammelt, mit gleich
vielen Bildern je Klasse. `probe_ids` wählt deshalb jeweils den Anfang einer
solchen Gruppe: 0, 5.500, 11.000 und so weiter.
Ein einzelnes Bild hat die Form `(1, 28, 28)`, der Bildbatch `(10, 1, 28, 28)`.
Die erste Dimension zählt Bilder, die zweite Farbkanäle. `y_probe` hat nur
die Form `(10,)`, weil pro Bild genau eine ganzzahlige Ziel-ID vorliegt.

Zum Anzeigen rechnen wir die Normalisierung mit `Bild * 0.5 + 0.5` zurück.
`[0]` wählt den einzigen Graustufenkanal; es schneidet keinen Bildbereich ab.
Die Titel zeigen die **wahren Labels**, noch keine Modellvorhersagen.
Schätzen Sie vor der Ausführung: Welche Kleidungsarten könnten bei dieser
Auflösung schwer zu unterscheiden sein?

In [ ]:
probe_ids = [k * (len(train_daten) // len(KLASSEN)) for k in range(len(KLASSEN))]
x_probe = torch.stack([train_daten[i][0] for i in probe_ids])
y_probe = torch.tensor([train_daten[i][1] for i in probe_ids], dtype=torch.long)
print("Bilder:", tuple(x_probe.shape), x_probe.dtype)
print("Ziele: ", tuple(y_probe.shape), y_probe.dtype)

fig, achsen = plt.subplots(2, 5, figsize=(11, 5))
for ax, bild, ziel in zip(achsen.flat, x_probe, y_probe):
    ax.imshow((bild[0] * 0.5 + 0.5).numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(KLASSEN[ziel.item()])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Aus einem Bild wird eine Sequenz

### 3.1 Patchzerlegung: keine IDs, keine verlorenen Pixel

Ein Texttoken wird durch eine diskrete ID bezeichnet. Ein Bildpatch ist hier
direkt durch seine **16 Pixelwerte** beschrieben. Wir trainieren keinen
BPE-Tokenizer und legen kein Bildvokabular an.

`zerlege_patches` erwartet `(B, C, H, W)`: Batchgröße, Kanäle, Höhe, Breite.
Für die Patchkante `p = 4` schreibt `reshape` die räumlichen Achsen zunächst
als `(B, C, H/p, p, W/p, p)`. Damit sind Rasterplätze und Pixel innerhalb
eines Patches getrennt. `permute(0, 2, 4, 1, 3, 5)` ordnet sie zu
`(B, Rasterzeile, Rasterspalte, C, Patchzeile, Patchspalte)` um.
Das abschließende `reshape` fasst Rasterplätze sowie Patchpixel zusammen:
`(B, 49, 16)`. Die Rasterreihenfolge ist zeilenweise von links oben nach rechts
unten; innerhalb eines Graustufenpatches ebenfalls zeilenweise.

Diese Operation sortiert Werte nur um. Sie lernt nichts, skaliert nichts und
verwirft keine Randpixel. Nicht teilbare Bildkanten werden ausdrücklich abgewiesen.

In [ ]:
def zerlege_patches(bilder, patch_size):
    if bilder.ndim != 4:
        raise ValueError("Erwartet wird ein Bildbatch (B, C, H, W).")
    if type(patch_size) is not int or patch_size <= 0:
        raise ValueError("patch_size muss eine positive ganze Zahl sein.")
    B, C, H, W = bilder.shape
    p = patch_size
    if H % p or W % p:
        raise ValueError("Bildkanten müssen durch patch_size teilbar sein.")
    x = bilder.reshape(B, C, H // p, p, W // p, p)
    x = x.permute(0, 2, 4, 1, 3, 5)
    return x.reshape(B, (H // p) * (W // p), C * p * p)


patch_probe = zerlege_patches(x_probe, konfig.patch_size)
print("Vorher:", tuple(x_probe.shape))
print("Nachher:", tuple(patch_probe.shape))

### Beobachtung: derselbe Inhalt als Bild und als Patchraster

Links zeigt die nächste Zelle das erste Bild mit Rastergrenzen bei jeder
vierten Pixelkante. Rechts setzt sie **die tatsächlich zerlegten Patches** wieder
an ihre Rasterplätze. Die Nummer `i` ist der Platz in der Patchsequenz, nicht
die ID eines gelernten Vokabulareintrags. Die Anzeigen werden nur für die
Darstellung vergrößert; die Eingabedaten bleiben 28 × 28 Pixel groß.

`divmod(i, rasterkante)` liefert Rasterzeile und Rasterspalte. Vergleichen Sie
vor allem Übergänge über Patchgrenzen: Der Bildinhalt muss erhalten bleiben.

In [ ]:
rasterkante = konfig.image_size // konfig.patch_size
fig = plt.figure(figsize=(10, 5))
links, rechts = fig.subfigures(1, 2)
ax = links.subplots()
ax.imshow((x_probe[0, 0] * 0.5 + 0.5).numpy(), cmap="gray", vmin=0, vmax=1)
for grenze in range(konfig.patch_size, konfig.image_size, konfig.patch_size):
    ax.axhline(grenze - 0.5, color="orange", lw=0.8)
    ax.axvline(grenze - 0.5, color="orange", lw=0.8)
ax.set_title("Original mit Patchgrenzen")
ax.axis("off")
achsen = rechts.subplots(rasterkante, rasterkante)
for i, patch in enumerate(patch_probe[0]):
    zeile, spalte = divmod(i, rasterkante)
    ax = achsen[zeile, spalte]
    bild = patch.reshape(konfig.patch_size, konfig.patch_size) * 0.5 + 0.5
    ax.imshow(bild.numpy(), cmap="gray", vmin=0, vmax=1)
    ax.text(0, 0, str(i), color="orange", fontsize=6)
    ax.axis("off")
rechts.suptitle("Die Patchsequenz an ihren Rasterplätzen")
plt.show()

### 3.2 Patch-Embedding: aus 16 Pixelwerten werden 128 Merkmale

`PatchEmbedding` verbindet die feste Zerlegung mit einer lernbaren Projektion.
`nn.Linear(16, 128)` berechnet für jeden Patch 128 gewichtete Summen plus Bias.
**Alle 49 Patches verwenden dieselbe Projektion.** Dadurch hängt der anfängliche
Inhaltsvektor von den Pixelwerten ab, nicht vom Rasterplatz.

Die Breite 128 entspricht `d_model` aus GuppyLM. Anders als `nn.Embedding`
schlägt die Projektion keine Tabellenzeile über eine ID nach: Sie berechnet den
Vektor direkt aus kontinuierlichen Pixelwerten. Ihre Gewichte werden gemeinsam
mit dem restlichen Modell gelernt. `forward` prüft die vereinbarte Bildform;
es gibt kein stilles Anpassen der Auflösung.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.konfig = konfig
        pixel_pro_patch = konfig.in_channels * konfig.patch_size ** 2
        self.proj = nn.Linear(pixel_pro_patch, konfig.d_model)

    def forward(self, bilder):
        cfg = self.konfig
        erwartet = (cfg.in_channels, cfg.image_size, cfg.image_size)
        if bilder.ndim != 4 or tuple(bilder.shape[1:]) != erwartet:
            raise ValueError(f"Erwartet wird (B, {erwartet[0]}, {erwartet[1]}, {erwartet[2]}).")
        patches = zerlege_patches(bilder, cfg.patch_size)
        return self.proj(patches)

### 3.3 Classification Token und Positions-Embeddings

Für das ganze Bild brauchen wir am Ausgang einen zusammenfassenden Vektor.
Wir stellen deshalb einen trainierbaren Startvektor **`[CLS]`** vor die Patches.
Er enthält anfangs keine Bildpixel und ist für alle Bilder derselbe Parameter.
Durch die folgenden Attentionblöcke kann er Bildinformationen aufnehmen.
Er ist selbst keine Klassen-ID. Erst ein späterer Kopf liest daraus zehn Scores.

`BildSequenz` besitzt `cls_token` mit Form `(1, 1, d_model)` und `pos_emb`
mit Form `(1, seq_len, d_model)`. `nn.Parameter` macht beide zu trainierbaren
Modellparametern. Die kleinen normalverteilten Startwerte werden mit
`trunc_normal_` initialisiert. `expand(B, -1, -1)` verwendet denselben
CLS-Parameter für jedes Bild. `torch.cat(..., dim=1)` setzt ihn vor die Patches.

Wie im GuppyLM addieren wir anschließend ein gelerntes Positions-Embedding.
Position 0 gehört `[CLS]`; Positionen 1 bis 49 bezeichnen die festen Plätze
im Patchraster. Auch eine eindimensionale Tabelle kann dieses feste Raster
abbilden. Ohne Positionsinformation könnte die folgende globale Self-Attention
die Reihenfolge der Patchvektoren nicht unterscheiden. Zum Schluss wirkt
Dropout auf die Summe, während `eval()` es abschaltet.

In [ ]:
class BildSequenz(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.patch_emb = PatchEmbedding(konfig)
        self.cls_token = nn.Parameter(torch.empty(1, 1, konfig.d_model))
        self.pos_emb = nn.Parameter(torch.empty(1, konfig.seq_len, konfig.d_model))
        self.drop = nn.Dropout(konfig.dropout)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward(self, bilder):
        patches = self.patch_emb(bilder)
        cls = self.cls_token.expand(bilder.shape[0], -1, -1)
        x = torch.cat([cls, patches], dim=1)
        return self.drop(x + self.pos_emb)

## 4. Der Transformer

### 4.1 Multi-Head Self-Attention — bekannte Rechnung, andere Sichtbarkeit

Der Eingang hat die Form `(B, T, C)`: Bilder im Batch, Sequenzpositionen und
Modellbreite. `T` enthält `[CLS]` und alle Bildpatches; `C` ist `d_model`.
Wie im GuppyLM berechnet `qkv` mit einer linearen Abbildung Q, K und V für
sämtliche Köpfe. **Jeder Kopf verwendet den gesamten Eingangsvektor.** Erst die
projizierten Werte werden in `n_heads` Teilvektoren mit Breite `head_dim` zerlegt.

Die folgende Zelle rechnet genau diese Schritte:

1. `reshape` und `permute` bringen Q/K/V auf `(B, Köpfe, T, head_dim)`.
2. `q @ k.transpose(-2, -1)` vergleicht jede Query mit jedem Key.
   Division durch `sqrt(head_dim)` skaliert die Scores.
3. `softmax(..., dim=-1)` normiert **pro Query über die Keypositionen**.
4. Die Gewichte mischen die Values; die Köpfe werden zusammengeführt und mit
   `out` auf die Modellbreite projiziert.

Es fehlt bewusst `masked_fill` aus GuppyLM. Das ganze Bild ist Eingabe, nicht
ein zu erzeugendes nächstes Bildstück. Jeder Patch und `[CLS]` dürfen alle
Positionen einschließlich sich selbst verwenden. Die Klassen-ID wird an keiner
Stelle als Eingabe hineingegeben. Alle Bilder sind gleich groß; Padding ist
hier ebenfalls unnötig.

`return_attn=True` liefert zusätzlich die Gewichte **vor dem Dropout** zurück.
Diese Zeilen summieren sich zu 1. Im Trainingszweig kann Dropout einige Gewichte
nullsetzen und die übrigen skalieren; danach gilt diese Zeilensumme nicht mehr
für jeden einzelnen Durchlauf. Im Auswertungsmodus bleibt die Matrix unverändert.

In [ ]:
class Attention(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.n_heads = konfig.n_heads
        self.head_dim = konfig.d_model // konfig.n_heads
        self.qkv = nn.Linear(konfig.d_model, 3 * konfig.d_model)
        self.out = nn.Linear(konfig.d_model, konfig.d_model)
        self.dropout = nn.Dropout(konfig.dropout)

    def forward(self, x, return_attn=False):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        roh = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        gewichte = F.softmax(roh, dim=-1)
        y = self.dropout(gewichte) @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out(y)
        return (y, gewichte) if return_attn else y

### 4.2 FFN — jede Position getrennt weiterverarbeiten

Attention mischt Informationen **zwischen Positionen**. Das FFN verarbeitet
anschließend jede Position für sich, mit denselben Gewichten an allen Positionen.
`hoch` erweitert 128 auf 512 Komponenten, GELU wirkt als nichtlineare Funktion,
`runter` führt auf 128 Komponenten zurück. Diese gleiche Ein- und Ausgangsbreite
ermöglicht gleich die Residualaddition. Das FFN ist noch nicht der
Klassifikationskopf und erzeugt keine zehn Klassenwerte.

Die Namen und die GELU-Einstellung `approximate="tanh"` entsprechen GuppyLM.
Der folgende Code fügt abschließend Dropout hinzu; die Form `(B, T, C)` bleibt
insgesamt erhalten.

In [ ]:
class FFN(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.hoch = nn.Linear(konfig.d_model, konfig.ffn_hidden)
        self.runter = nn.Linear(konfig.ffn_hidden, konfig.d_model)
        self.dropout = nn.Dropout(konfig.dropout)

    def forward(self, x):
        x = self.hoch(x)
        x = F.gelu(x, approximate="tanh")
        return self.dropout(self.runter(x))

### 4.3 Der Block — dieselben zwei Residualadditionen

Ein Block enthält Attention und FFN. Er berechnet wie GuppyLM:

\[
u=x+\operatorname{Attention}(\operatorname{LayerNorm}_1(x)),\qquad
y=u+\operatorname{FFN}(\operatorname{LayerNorm}_2(u)).
\]

`norm1` normalisiert für jede Position über die 128 Komponenten, bevor Q/K/V
berechnet werden. Die erste Addition erhält zugleich den unveränderten Eingang
über den Residualpfad. `norm2` bereitet danach den FFN-Zweig vor; die zweite
Addition erhält dessen Eingang. **Pre-Norm** bezeichnet diese Lage der
Normierungen vor den verarbeitenden Teilnetzen.

Im folgenden Code verwendet die zweite Zeile bereits das durch die erste Zeile
aktualisierte `x`. Positions-Embeddings werden im Block nicht erneut addiert.
`Block` erhält im Gegensatz zum GuppyLM keine kausale Maske.

In [ ]:
class Block(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.norm1 = nn.LayerNorm(konfig.d_model)
        self.attn = Attention(konfig)
        self.norm2 = nn.LayerNorm(konfig.d_model)
        self.ffn = FFN(konfig)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

### 4.4 Das vollständige Modell und die zehn Logits

`VisionTransformer` setzt die Bausteine zusammen. `eingang` erzeugt die
Bildsequenz; `ModuleList` hält vier nacheinander durchlaufene Blöcke mit
**je eigenen Gewichten**. `norm` ist wie im GuppyLM eine abschließende LayerNorm.
Bis hier bleiben alle 50 Positionen und ihre 128 Komponenten erhalten.

Jetzt unterscheidet sich der Ausgang: `x[:, 0, :]` wählt für jedes Bild den
fertigen CLS-Vektor. `kopf = nn.Linear(d_model, n_classes)` macht daraus zehn
**Logits**, also unnormierte Klassenscores. Die Rückgabe hat `(B, 10)` statt
`(B, T, Vokabulargröße)`. Es gibt keine Bindung der Kopfgewichte an die
Eingabeprojektion: Pixelwerte und Klassen sind verschiedene Größen.

Falls `ziele` mitgegeben werden, berechnet `cross_entropy(logits, ziele)` den
Trainingsverlust. Diese Funktion erwartet **rohe Logits**; sie enthält die
nötige Log-Softmax-Rechnung bereits. Eine zusätzliche Softmax davor wäre falsch.
Ohne Ziele liefert das Modell `None` als Verlust. Das Rückgabepaar
`(logits, verlust)` bleibt damit vom GuppyLM vertraut.

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.konfig = konfig
        self.eingang = BildSequenz(konfig)
        self.blocks = nn.ModuleList([Block(konfig) for _ in range(konfig.n_layers)])
        self.norm = nn.LayerNorm(konfig.d_model)
        self.kopf = nn.Linear(konfig.d_model, konfig.n_classes)

    def forward(self, bilder, ziele=None):
        x = self.eingang(bilder)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        logits = self.kopf(x[:, 0, :])
        verlust = F.cross_entropy(logits, ziele) if ziele is not None else None
        return logits, verlust

### Beobachtung: ein vollständiger Vorwärtslauf vor dem Lernen

`probe_modell` ist ein frisches, untrainiertes Modell auf der CPU. `eval()`
schaltet Dropout ab; `torch.no_grad()` verhindert das Aufbauen eines
Gradientengraphen. Beide Einstellungen erfüllen verschiedene Aufgaben.
`softmax` verwenden wir jetzt nur zur **Anzeige** der Klassenverteilung.
`argmax` wählt die größte Komponente. Vor dem Lernen ist dies noch keine
brauchbare Erkennung; die Zahlen entstehen aus zufälligen Gewichten.

`sum(p.numel() ...)` zählt alle Modellparameter. Der Verlust einer exakt
gleichmäßigen Verteilung wäre `ln(10)`. Das ist eine Referenz und kein
garantierter Startverlust. Die Gleichverteilung liefert jeder Klasse 10 %;
ein hoher einzelner Modellscore belegt für sich keine zuverlässige Vorhersage.

In [ ]:
probe_modell = VisionTransformer(konfig).eval()
with torch.no_grad():
    logits, verlust = probe_modell(x_probe, y_probe)
    wahrscheinlichkeiten = logits.softmax(dim=-1)
print(f"Parameter: {sum(p.numel() for p in probe_modell.parameters()):,}")
print("Logits:", tuple(logits.shape))
print(f"Untrainierter Verlust: {verlust.item():.4f}; ln(10): {math.log(10):.4f}")
for klasse, wert in zip(KLASSEN, wahrscheinlichkeiten[0]):
    print(f"{klasse:14s}: {wert.item():6.2%}")

### Beobachtung: `[CLS]` darf auch den letzten Patch sehen

Wir berechnen zunächst den Eingang von Block 1 und dessen `norm1`. Genau diese
normalisierten Vektoren erhält die erste Attention. `return_attn=True` gibt
die Matrix aller Köpfe zurück. Für ein Bild hat sie die Form `(Köpfe, 50, 50)`.

Die Zelle zeigt Kopf 1: **Zeile = fragende Position, Spalte = gelesene Position**.
Zeile und Spalte 0 gehören `[CLS]`. Rechts oberhalb der Diagonalen steht hier
keine gesperrte Dreieckshälfte. `gewichte[0, 0, 0, -1]` liest explizit das Gewicht
vom CLS-Token zum letzten Patch. Die Zeilensummen sollten bis auf
Rundungsfehler 1 sein. Diese Beobachtung beschreibt die Sichtbarkeit vor dem
Training; sie erklärt noch keine gelernte Klassifikationsentscheidung.

In [ ]:
with torch.no_grad():
    eingang = probe_modell.eingang(x_probe[:1])
    normalisiert = probe_modell.blocks[0].norm1(eingang)
    _, gewichte = probe_modell.blocks[0].attn(normalisiert, return_attn=True)
matrix = gewichte[0, 0]
print("Attentionform:", tuple(gewichte.shape))
print("Zeilensummen min/max:", matrix.sum(-1).min().item(), matrix.sum(-1).max().item())
print("[CLS] → letzter Patch:", matrix[0, -1].item())
plt.figure(figsize=(6, 5))
plt.imshow(matrix.numpy(), cmap="viridis")
plt.xlabel("Gelesene Position (Key)")
plt.ylabel("Fragende Position (Query)")
plt.title("Untrainiert: Block 1, Kopf 1; Position 0 = [CLS]")
plt.colorbar(label="Attentiongewicht")
plt.tight_layout()
plt.show()

## 5. Lernen

### Lernrate und Zahl der Updates

Wir trainieren zunächst `EPOCHEN = 15` vollständige Durchläufe. Das ist ein
Ausgangswert für die Erprobung auf der vorhandenen Hardware. Pro Epoche gibt es
`len(train_loader)` Updates; der letzte Batch darf kleiner als 128 sein.
`MAX_STEPS` ist das Produkt aus Epochen und Updates pro Epoche.

`LR` ist die größte Lernrate, `MIN_LR` die kleinste. `lernrate(schritt)`
verwendet nullbasierte Schritte: Im **Warm-up** wächst die Lernrate während
der ersten Epoche linear bis `LR`. Danach sinkt sie mit einer halben
Kosinuskurve bis `MIN_LR`. Die Sonderfälle verhindern eine Division durch null
bei sehr kurzen Probeläufen.

`WEIGHT_DECAY` steuert später den Gewichtszerfall in AdamW. `GRAD_CLIP` ist die
maximale Gesamtnorm der Gradienten vor einem Update. Ein großer Datensatz allein
garantiert keine schnelle Konvergenz; deshalb protokollieren wir Zeit und
Validierung und versprechen hier keine feste Genauigkeit.

In [ ]:
EPOCHEN = 15
LR = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
MAX_STEPS = EPOCHEN * len(train_loader)
WARMUP_STEPS = min(len(train_loader), max(0, MAX_STEPS - 1))
CHECKPOINT_PFAD = WORK / "Checkpoints" / "fashion_vit_bestes_modell.pt"

def lernrate(schritt):
    if not 0 <= schritt < MAX_STEPS:
        raise ValueError("Schritt liegt außerhalb des Trainingsplans.")
    if WARMUP_STEPS and schritt < WARMUP_STEPS:
        return LR * (schritt + 1) / WARMUP_STEPS
    fortschritt = (schritt - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS - 1)
    return MIN_LR + 0.5 * (LR - MIN_LR) * (1 + math.cos(math.pi * fortschritt))

print(f"{len(train_loader)} Updates/Epoche; insgesamt {MAX_STEPS} Updates")

### Verlust und Trefferquote ohne Gewichtsänderung messen

`bewerte(modell, loader)` kann später sowohl Validierungs- als auch Testdaten
auswerten. Die Funktion merkt sich zunächst den bisherigen Modus,
schaltet mit `eval()` Dropout ab und stellt den Modus zum Schluss wieder her.
Der Dekorator `@torch.no_grad()` verhindert Gradientenberechnung. Es gibt hier
weder `backward()` noch ein Optimiererupdate.

Für jeden Batch werden Bilder und Ziele auf dasselbe Gerät wie das Modell
gelegt. Der Batchverlust ist ein Mittelwert. Wir multiplizieren ihn deshalb
mit der tatsächlichen Bildzahl und teilen am Ende durch die Gesamtzahl.
So erhält auch ein kleiner letzter Batch das richtige Gewicht.
`logits.argmax(-1)` liefert pro Bild die Klasse mit dem größten Score; der
Vergleich mit `y` zählt Treffer. Die Rückgabe enthält mittleren Verlust und
Trefferanteil zwischen 0 und 1. Ein leerer Loader wird ausdrücklich abgewiesen.

In [ ]:
@torch.no_grad()
def bewerte(modell, loader):
    war_training = modell.training
    device = next(modell.parameters()).device
    modell.eval()
    verlustsumme, treffer, anzahl = 0.0, 0, 0
    try:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, verlust = modell(x, y)
            verlustsumme += verlust.item() * y.numel()
            treffer += (logits.argmax(-1) == y).sum().item()
            anzahl += y.numel()
    finally:
        modell.train(war_training)
    if anzahl == 0:
        raise ValueError("Der Auswertungsloader ist leer.")
    return verlustsumme / anzahl, treffer / anzahl

### Die Gewichte lernen lassen

Die nächste Zelle beginnt **jedes Mal ein neues Training**. Sie setzt den Seed
zurück, baut die Loader neu und erzeugt einen frischen `VisionTransformer`.
Die vorherigen Beobachtungen an `probe_modell` beeinflussen diesen Start nicht.
`AdamW` erhält alle Modellparameter, einschließlich CLS- und Positionsvektoren.
Der bisherige Dateistand am angegebenen Checkpointpfad wird bei der ersten
Verbesserung ersetzt; bewahren Sie Vergleichsläufe unter anderen Namen auf.

Für jeden Batch geschieht in dieser Reihenfolge:

1. `zero_grad` entfernt alte Gradienten; die aktuelle Lernrate wird eingetragen.
2. `modell(x, y)` berechnet Vorhersagen und Kreuzentropie.
3. `backward()` berechnet die Gradienten durch **alle** Bausteine und Residualpfade.
4. `clip_grad_norm_` begrenzt deren Gesamtnorm. Das begrenzt nicht unmittelbar
   die Länge des anschließenden AdamW-Gewichtsschritts.
5. `optimierer.step()` ändert die Gewichte.

Trainingsverlust und Trainingsgenauigkeit werden über alle Bilder der Epoche
gewichtet gemittelt. Diese Werte entstehen mit aktivem Dropout und während sich
die Gewichte ändern. Die anschließende Validierung verwendet ohne Dropout den
Stand am Epochenende. Die Messungen sind deshalb nicht unter identischen
Bedingungen entstanden.

`verlauf` sammelt die Messwerte. Sinkt der Validierungsverlust unter den bisher
besten, speichern wir Gewichte, Konfiguration, Klassenreihenfolge,
Vorverarbeitung, Splitindizes und Paketversionen. Der Testsatz wird dabei nicht
geladen. Die Datei dient der späteren Inferenz; für ein exaktes Fortsetzen
fehlen unter anderem Optimierer- und Zufallsgeneratorzustände.

In [ ]:
torch.manual_seed(SEED)
train_loader, val_loader = baue_loader(train_daten, val_daten, BATCH_SIZE)
modell = VisionTransformer(konfig).to(DEVICE)
optimierer = torch.optim.AdamW(modell.parameters(), lr=LR,
                               weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
verlauf = []
bester_val = float("inf")
schritt = 0
t0 = time.perf_counter()

print("Epoche | Trainverlust | Valverlust | Trainquote | Valquote | Sekunden")
for epoche in range(1, EPOCHEN + 1):
    modell.train()
    verlustsumme, treffer, anzahl = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimierer.zero_grad(set_to_none=True)
        for gruppe in optimierer.param_groups:
            gruppe["lr"] = lernrate(schritt)
        logits, verlust = modell(x, y)
        if not torch.isfinite(verlust):
            raise RuntimeError("Nicht endlicher Trainingsverlust; Lauf abgebrochen.")
        verlust.backward()
        torch.nn.utils.clip_grad_norm_(modell.parameters(), GRAD_CLIP,
                                       error_if_nonfinite=True)
        optimierer.step()
        verlustsumme += verlust.item() * y.numel()
        treffer += (logits.argmax(-1) == y).sum().item()
        anzahl += y.numel()
        schritt += 1

    val_verlust, val_quote = bewerte(modell, val_loader)
    messung = {"epoche": epoche, "train_verlust": verlustsumme / anzahl,
               "train_quote": treffer / anzahl, "val_verlust": val_verlust,
               "val_quote": val_quote, "sekunden": time.perf_counter() - t0}
    verlauf.append(messung)
    print(f"{epoche:6d} | {messung['train_verlust']:12.4f} | {val_verlust:10.4f} | "
          f"{messung['train_quote']:10.2%} | {val_quote:8.2%} | {messung['sekunden']:8.1f}")

    if val_verlust < bester_val:
        bester_val = val_verlust
        checkpoint = {
            "format_version": 1, "datensatz": "FashionMNIST",
            "konfig": asdict(konfig), "model_state_dict": modell.state_dict(),
            "klassen": list(KLASSEN), "vorverarbeitung": dict(VORVERARBEITUNG),
            "seed": SEED, "train_ids": list(train_ids), "val_ids": list(val_ids),
            "epoche": epoche, "schritte": schritt,
            "val_verlust": val_verlust, "val_quote": val_quote,
            "torch_version": str(torch.__version__),
            "torchvision_version": __import__("torchvision").__version__,
            "verlauf": list(verlauf),
        }
        torch.save(checkpoint, CHECKPOINT_PFAD)

print(f"Bester Validierungsverlust: {bester_val:.4f}")
print(f"Checkpoint: {CHECKPOINT_PFAD}")

### Beobachtung: Lernen und Überanpassung

Die nächste Zelle zeichnet die gespeicherten Epochenmittel: links
Kreuzentropie, rechts Trefferquote. Die gestrichelte Linie `ln(10)` ist wieder
die Verlustreferenz für eine Gleichverteilung. Sinkender Trainingsverlust bei
steigendem Validierungsverlust kann auf Überanpassung hinweisen.

Eine höhere Validierungsquote als Trainingsquote ist wegen Dropout und der
unterschiedlichen Messzeitpunkte nicht automatisch ein Fehler. Betrachten Sie
beide Kurven zusammen. Die Checkpointauswahl verwendet den **Validierungsverlust**;
die Epoche mit der höchsten Trefferquote muss nicht dieselbe sein.

In [ ]:
epochen = [m["epoche"] for m in verlauf]
fig, achsen = plt.subplots(1, 2, figsize=(11, 4))
for name, titel, ax in zip(("verlust", "quote"), ("Kreuzentropie", "Trefferanteil"), achsen):
    ax.plot(epochen, [m[f"train_{name}"] for m in verlauf], "o-", label="Training")
    ax.plot(epochen, [m[f"val_{name}"] for m in verlauf], "o-", label="Validierung")
    ax.set_xlabel("Epoche")
    ax.set_ylabel(titel)
    ax.legend()
achsen[0].axhline(math.log(10), color="gray", ls="--", label="ln(10)")
achsen[0].legend()
achsen[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 6. Prüfen und Anwenden

### Den über die Validierung ausgewählten Stand laden

`lade_checkpoint` liest den gespeicherten Stand zunächst auf die CPU.
`weights_only=True` beschränkt das Laden auf den unterstützten Daten- und
Tensorumfang; laden Sie dennoch nur Checkpoints aus vertrauenswürdiger Quelle.
Die Funktion prüft Format, Datensatzname, Klassenliste und Pixelumrechnung.
Die Architektur entsteht aus der **gespeicherten** Konfiguration, nicht aus
möglicherweise inzwischen geänderten Einstellungen im Notebook.

`load_state_dict` verlangt passende Gewichte. Anschließend versetzt `eval()` das
Netz in den Auswertungsmodus. Wir verwenden ab hier `bestes_modell` für den
ausgewählten Stand, auch wenn die letzte Trainingsepoche schlechter war.
Die Prüfung der Pixelumrechnung verhindert, dass wir unbemerkt anders
vorverarbeiten als während des Trainings.

In [ ]:
def lade_checkpoint(pfad, device):
    ckpt = torch.load(pfad, map_location="cpu", weights_only=True)
    if ckpt.get("format_version") != 1 or ckpt.get("datensatz") != "FashionMNIST":
        raise ValueError("Kein unterstützter Fashion-MNIST-ViT-Checkpoint.")
    if ckpt.get("klassen") != KLASSEN or ckpt.get("vorverarbeitung") != VORVERARBEITUNG:
        raise ValueError("Klassenreihenfolge oder Vorverarbeitung passen nicht.")
    cfg = ViTConfig(**ckpt["konfig"])
    if (cfg.image_size, cfg.in_channels, cfg.n_classes) != (28, 1, 10):
        raise ValueError("Die Checkpointkonfiguration passt nicht zu Fashion-MNIST.")
    netz = VisionTransformer(cfg)
    netz.load_state_dict(ckpt["model_state_dict"])
    netz = netz.to(device).eval()
    return netz, ckpt


bestes_modell, ckpt = lade_checkpoint(CHECKPOINT_PFAD, DEVICE)
print(f"Gewählt: Epoche {ckpt['epoche']}; Validierungsquote {ckpt['val_quote']:.2%}")

### Abschlusstest: jetzt erst die offiziellen 10.000 Testbilder

`train=False` öffnet den getrennten offiziellen Testsatz. Wir verwenden dieselbe
`vorverarbeitung` und die Funktion `bewerte`. Es findet kein Gewichtsupdate statt.
Die Ausgabe nennt Verlust und Trefferquote über den **vollständigen** Testsatz.

Der Testsatz beantwortet eine andere Frage als die Validierung: Wie gut ist das
bereits ausgewählte Modell auf bisher unbenutzten Testdaten? Ändern wir nach
diesem Ergebnis immer wieder Architektur oder Lernrate, werden die Testdaten
indirekt zur Modellwahl verwendet. Für solche Experimente treffen wir die Wahl
zunächst anhand der Validierung und führen den Abschlusstest erst danach aus.

In [ ]:
test_daten = FashionMNIST(root=WORK / "Data", train=False,
                         download=HERUNTERLADEN, transform=vorverarbeitung)
test_loader = DataLoader(test_daten, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_verlust, test_quote = bewerte(bestes_modell, test_loader)
print(f"Abschlusstest: {len(test_daten):,} Bilder")
print(f"Verlust: {test_verlust:.4f}; Trefferquote: {test_quote:.2%}")

### Ein Bild klassifizieren: zehn Möglichkeiten, eine Entscheidung

`klassifiziere` erwartet einen bereits mit derselben Vorverarbeitung erzeugten
Bildbatch. `eval()` und `no_grad()` schalten Dropout und Gradientenaufzeichnung
ab. Erst **nach** dem Modell berechnet Softmax die zehn Klassenwahrscheinlichkeiten.
`max(dim=-1)` liefert für jedes Bild den größten Wert und dessen Klassen-ID.
Die Ausgaben werden zur weiteren Anzeige auf die CPU zurückgebracht.

Hier ist die Analogie zum Sprachmodell besonders eng: Dort wählen wir am
Sprachmodellkopf ein Token aus dem Vokabular, hier am Klassifikationskopf eine
Klasse. Der gewählte Klassenname wird aber nicht an die Eingabe angehängt.
Ein Vorwärtslauf genügt; es gibt keine autoregressive Generierungsschleife.
Der Softmaxwert ist eine Modellwahrscheinlichkeit, keine zugesicherte
Häufigkeit richtiger Entscheidungen. Dafür wäre zusätzlich Kalibrierung zu prüfen.

In [ ]:
@torch.no_grad()
def klassifiziere(modell, bilder):
    modell.eval()
    device = next(modell.parameters()).device
    logits, _ = modell(bilder.to(device))
    wahrscheinlichkeiten = logits.softmax(dim=-1)
    scores, klassen = wahrscheinlichkeiten.max(dim=-1)
    return klassen.cpu(), scores.cpu(), wahrscheinlichkeiten.cpu()

### Beobachtung: Vorhersagen und Fehler sichtbar machen

Die nächste Zelle wählt die **ersten zwölf** Testbilder, ohne sie nach schönen
Ergebnissen auszusuchen. `ist` zeigt das wahre Label; `Modell` die Vorhersage
und ihren Softmaxwert. Grün bedeutet richtig, Rot falsch. Eine Bildauswahl kann
keinen vollständigen Qualitätsnachweis ersetzen; dafür steht die vorherige
Messung über alle Testbilder.

Unter den Bildern erscheinen die drei größten Klassenwerte für das erste Bild.
`topk(3)` sortiert diese drei Werte; es verändert die Klassifikation nicht.
Vergleichen Sie vor der Ausführung Ihre eigene Vermutung mit der späteren
Verteilung. Welche Alternativklasse ist plausibel?

In [ ]:
bilder = torch.stack([test_daten[i][0] for i in range(12)])
ziele = torch.tensor([test_daten[i][1] for i in range(12)])
vorhersagen, scores, verteilungen = klassifiziere(bestes_modell, bilder)
fig, achsen = plt.subplots(3, 4, figsize=(12, 9))
for ax, bild, ziel, geraten, score in zip(achsen.flat, bilder, ziele, vorhersagen, scores):
    ax.imshow((bild[0] * 0.5 + 0.5).numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Ist: {KLASSEN[ziel.item()]}\nModell: {KLASSEN[geraten.item()]} ({score:.0%})",
                 color="green" if ziel == geraten else "red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
werte, ids = verteilungen[0].topk(3)
print("Drei größte Klassenwerte für das erste Bild:")
for klasse, wert in zip(ids, werte):
    print(f"{KLASSEN[klasse.item()]:14s}: {wert.item():.2%}")

### Welche Klassen werden miteinander verwechselt?

Die Konfusionsmatrix enthält **Zeilen für wahre Klassen** und **Spalten für
vorhergesagte Klassen**. Diagonaleinträge sind Treffer. `y * 10 + geraten`
kodiert jedes Klassenpaar als Zahl zwischen 0 und 99. `bincount` zählt diese
Paare; `reshape(10, 10)` stellt sie als Matrix dar. Dies ist eine zweite
Auswertung desselben unveränderten Testmodells, kein neuer Trainingsschritt.

`quoten` teilt die Diagonale durch die Zahl wahrer Bilder jeder Klasse.
So sehen wir, ob hinter einer guten Gesamtquote eine schwächere Einzelklasse
steckt. Die Matrix zeigt Anzahlen, keine Attentiongewichte und keine Logits.

In [ ]:
matrix = torch.zeros(len(KLASSEN), len(KLASSEN), dtype=torch.long)
for x, y in test_loader:
    geraten, _, _ = klassifiziere(bestes_modell, x)
    paare = y * len(KLASSEN) + geraten
    matrix += torch.bincount(paare, minlength=len(KLASSEN) ** 2).reshape(len(KLASSEN), -1)

fig, ax = plt.subplots(figsize=(9, 8))
bild = ax.imshow(matrix.numpy(), cmap="Blues")
ax.set_xticks(range(len(KLASSEN)), KLASSEN, rotation=60, ha="right")
ax.set_yticks(range(len(KLASSEN)), KLASSEN)
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")
for i in range(len(KLASSEN)):
    for j in range(len(KLASSEN)):
        ax.text(j, i, str(matrix[i, j].item()), ha="center", va="center", fontsize=7,
                color="white" if matrix[i, j] > matrix.max() / 2 else "black")
fig.colorbar(bild, ax=ax, label="Anzahl Bilder")
plt.tight_layout()
plt.show()
quoten = matrix.diag() / matrix.sum(dim=1).clamp_min(1)
for name, quote in zip(KLASSEN, quoten):
    print(f"{name:14s}: {quote.item():.2%}")

### Ein Blick in die gelernte Attention

Für ein Testbild rekonstruieren wir den Weg bis zum **letzten Block**: Eingang,
alle vorherigen Blöcke und dessen `norm1`. Die Attention des letzten Blocks
liefert dann ihre Gewichte. `a[0, :, 0, 1:]` wählt das erste Bild, alle Köpfe,
die CLS-Query und nur die 49 Patchspalten. Die CLS-Spalte selbst wird zur
Bilddarstellung weggelassen. Deshalb müssen diese 49 Werte **nicht** zusammen
1 ergeben. `reshape` bringt sie in das 7 × 7 große Raster zurück.

Wir zeigen jeden Kopf einzeln mit gemeinsamer Farbskala. Hellere Rasterplätze
erhalten in dieser Attentionrechnung mehr Gewicht. Die Karte ist **keine
vollständige Erklärung der Klassifikation**: Values, Ausgabeprojektionen, FFNs,
Residualpfade und frühere Blöcke tragen ebenfalls zur Entscheidung bei. Sie ist
auch keine Segmentierung oder Bounding Box.

In [ ]:
bestes_modell.eval()
cfg = bestes_modell.konfig
with torch.no_grad():
    x = bestes_modell.eingang(bilder[:1].to(DEVICE))
    for block in bestes_modell.blocks[:-1]:
        x = block(x)
    letzter = bestes_modell.blocks[-1]
    _, a = letzter.attn(letzter.norm1(x), return_attn=True)
    karten = a[0, :, 0, 1:].cpu().reshape(cfg.n_heads,
                                        cfg.image_size // cfg.patch_size,
                                        cfg.image_size // cfg.patch_size)

fig, achsen = plt.subplots(1, cfg.n_heads + 1, figsize=(3 * (cfg.n_heads + 1), 3))
achsen[0].imshow((bilder[0, 0] * 0.5 + 0.5).numpy(), cmap="gray", vmin=0, vmax=1)
achsen[0].set_title("Testbild")
for kopf, karte in enumerate(karten):
    achsen[kopf + 1].imshow(karte.numpy(), cmap="magma", vmin=0, vmax=karten.max().item())
    achsen[kopf + 1].set_title(f"Kopf {kopf + 1}: CLS → Patches")
for ax in achsen:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Weiterarbeiten: gezielt eine Eigenschaft verändern

Formulieren Sie vor jeder Änderung eine Erwartung und vergleichen Sie zunächst
anhand der **Validierung**. Verwenden Sie gleiche Splits und bewahren Sie
Checkpoints unter verschiedenen Namen auf.

- **Patchkante 7 statt 4:** Es entstehen 16 Bildtokens und ein CLS-Token.
  Das Pixelraster bleibt unverändert. Was ändert sich an der Attentionmatrix,
  der Projektionsbreite und der Trainingszeit?
- **Zwei statt vier Blöcke:** Breite und Kopfzahl bleiben gleich. Welche
  Rechenschritte und wie viele Parameter entfallen?
- **Ein statt vier Köpfe:** Halten Sie `d_model` gleich. Die Kopfbreite ändert
  sich, die Zahl der Q/K/V- und Ausgabeprojektionsparameter dadurch nicht.
- **Ohne Positions-Embeddings:** Trainieren Sie einen eigenen Vergleichslauf,
  indem in `BildSequenz.forward` die Addition von `self.pos_emb` entfällt.
  Eine Permutation ganzer Patchvektoren ist dann für den CLS-Ausgang bei
  deaktiviertem Dropout unsichtbar. Das bedeutet nicht, dass das Modell gar
  nichts lernen kann: Innerhalb eines Patches bleiben Pixelpositionen erhalten.

**Späteres Laden ohne erneutes Training:** In einer neuen Sitzung benötigen Sie
die Definitionen aus Abschnitt 1, die Klassenliste und Vorverarbeitung aus
Abschnitt 2, die Modellbausteine aus 3 und 4 sowie `lade_checkpoint` und
`klassifiziere` aus 6. Ein eigener Datenlader ist nur nötig, wenn Sie wieder
Fashion-MNIST-Bilder verwenden möchten. Einzelne Bilder müssen bereits die
vereinbarte Graustufenform und Pixelumrechnung besitzen.

**Quellen und Einordnung:** Die Architektur folgt dem klassischen
[Vision Transformer von Dosovitskiy et al.](https://arxiv.org/abs/2010.11929),
mit kleinerer Breite, Tiefe und Patchgröße für 28 × 28 große Bilder.
Benennungen und explizite Rechenschritte orientieren sich an unserem
`guppylm_lehrnotebook.ipynb`. Es werden keine vortrainierten Gewichte verwendet.
Fashion-MNIST stammt von Han Xiao, Kashif Rasul und Roland Vollgraf / Zalando
Research: [Originalprojekt, Daten und MIT-Lizenz](https://github.com/zalandoresearch/fashion-mnist),
[Datensatzarbeit](https://arxiv.org/abs/1708.07747).
Der [torchvision-Datenlader](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html)
übernimmt Download, Dateiprüfung und Lesen der Bilder. Diese Herkunftsangaben
beim Weitergeben des Materials beibehalten.